In [1]:
%pip -q install torchao==0.16.0
%pip install -q "transformers==4.57.1" "peft==0.17.1" "accelerate==1.11.0"

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
from importlib.metadata import version
from decimal import Decimal
import gc
import hashlib
import json
import re
import unicodedata

import pandas as pd
import torch
import transformers
from peft import PeftModel
from PIL import Image
from transformers import AutoProcessor, GenerationConfig, Qwen3VLForConditionalGeneration, set_seed

In [3]:
def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

MLLM_ROOT = Path("/content/drive/MyDrive/MLLM")
C_ROOT = MLLM_ROOT / "mordern_llm"
D_ROOT = MLLM_ROOT / "section_d"
DEVICE = torch.device("cuda:0")

E_ROOT = MLLM_ROOT / "section_e"
C_DATA = C_ROOT / "data/sroie"
PROCESSOR_DIR = C_ROOT / "outputs/qlora_run1/best_adapter"

PILOT = D_ROOT / "outputs/sft_pilot_v1"
RECIPE_PATH = PILOT / "recipe.json"
D_LOCK_PATH = D_ROOT / "data/heldout_manifest.json"
ADAPTER = PILOT / "llm_lora/best_adapter"

manifest = read_json(C_DATA / "manifest.json")
recipe = read_json(RECIPE_PATH)
CHECKPOINT_STEP = read_json(PILOT / "llm_lora/best_validation.json")["step"]

task = recipe["task"]
FIELDS = tuple(task["fields"])
DOC_PROMPT = task["prompt"]
EVAL_BUDGET = recipe["image_kwargs"]

DTYPE = {"torch.float16": torch.float16, "torch.bfloat16": torch.bfloat16}[recipe["dtype"]]

if DTYPE == torch.bfloat16 and torch.cuda.get_device_capability(DEVICE)[0] < 8:
    raise RuntimeError("Use a GPU with native BF16 support to match the saved recipe.")

processor = AutoProcessor.from_pretrained(PROCESSOR_DIR, use_fast=False)
processor.tokenizer.padding_side = "right"
E_ROOT.mkdir(parents=True, exist_ok=True)

print("Section C:", C_ROOT)
print("Section D:", D_ROOT)
print("Processor:", PROCESSOR_DIR)
print("LoRA checkpoint step:", CHECKPOINT_STEP)
print("Dtype:", DTYPE)

Section C: /content/drive/MyDrive/MLLM/mordern_llm
Section D: /content/drive/MyDrive/MLLM/section_d
Processor: /content/drive/MyDrive/MLLM/mordern_llm/outputs/qlora_run1/best_adapter
LoRA checkpoint step: 8
Dtype: torch.float16


In [4]:
def file_sha(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()

def write_json(path, data):
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def lock_json(path, data):
    if path.exists():
        assert read_json(path) == data, f"Existing lock differs: {path.name}"
    else:
        write_json(path, data)

def normal(field, value):
    if not isinstance(value, str) or not value.strip():
        return None

    text = " ".join(unicodedata.normalize("NFKC", value).casefold().split())
    if field != "total":
        return text
    
    text = re.sub(r"^(?:rm|\$)\s*", "", text)
    if not re.fullmatch(
        r"[+-]?(?:\d+|\d{1,3}(?:,\d{3})+)(?:\.\d+)?", text
    ):
        return None

    amount = Decimal(text.replace(",", ""))
    return format(amount.normalize(), "f")

In [5]:
def score_doc(answer, reference):
    raw_valid = True
    used_cleanup = False

    try:
        prediction = json.loads(answer)
    except json.JSONDecodeError:
        raw_valid = False
        prediction = None

        fence = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", answer.strip(), flags=re.IGNORECASE | re.DOTALL)

        if fence:
            used_cleanup = True
            try:
                prediction = json.loads(fence.group(1))
            except json.JSONDecodeError:
                pass

    schema_valid = (
        isinstance(prediction, dict)
        and set(prediction) == set(FIELDS)
        and all(value is None or isinstance(value, str) for value in prediction.values())
    )

    flags = {}

    for field in FIELDS:
        correct = False

        if schema_valid:
            predicted = prediction[field]
            expected = reference[field]

            if predicted is None or expected is None:
                correct = predicted is None and expected is None
            else:
                normalized = normal(field, predicted)
                correct = (
                    normalized is not None
                    and normalized == normal(field, expected)
                )

        flags[f"{field}_correct"] = correct

    return {
        "prediction": prediction,
        "raw_json_valid": raw_valid,
        "schema_valid": schema_valid,
        "used_fence_cleanup": used_cleanup,
        **flags,
        "all_correct": all(flags.values())
    }

In [6]:
CAPS = {"vqa": 16, "receipt": task["max_new_tokens"]}
adapter_hashes = [
    file_sha(ADAPTER / "adapter_config.json"),
    file_sha(ADAPTER / "adapter_model.safetensors")
]

protocol = {
    "base_id": recipe["model_id"],
    "base_revision": recipe["revision"],
    "adapter_path": str(ADAPTER),
    "adapter_hashes": adapter_hashes,
    "adapter_checkpoint_step": CHECKPOINT_STEP,
    "dtype": str(DTYPE),
    "quantization": None,
    "attention_implementation": "sdpa",
    "processor": str(PROCESSOR_DIR),
    "processor_use_fast": False,
    "image_kwargs": EVAL_BUDGET,
    "generation": {
        "max_new_tokens_by_task": CAPS,
        "do_sample": False,
        "use_cache": True,
        "use_model_defaults": False,
        "eos_token_id": processor.tokenizer.eos_token_id,
        "pad_token_id": processor.tokenizer.pad_token_id
    },
    "seed" : 42,
    "gpu" : torch.cuda.get_device_name(DEVICE),
    "versions": {name: version(name) for name in ("torch", "transformers", "peft", "accelerate")},
    "dev_manifest_sha256": file_sha(E_ROOT / "dev_samples_v1.json"),
    "scoring_rules": ["closed_choice_v1", "receipt_c_v1"],
    "failed_requests_in_denominator": True
}

RUN_ID = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode("utf-8")).hexdigest()[:12]

E_OUT = E_ROOT / "outputs" / RUN_ID
E_OUT.mkdir(parents=True, exist_ok=True)
lock_json(E_OUT / "protocol.json", protocol)

print("Run folder:", E_OUT)

Run folder: /content/drive/MyDrive/MLLM/section_e/outputs/ba2d7237c1e2


In [7]:
def load_eval_model(setting):
    if setting not in ("base", "llm_lora"):
        raise ValueError(f"Unknown evaluation setting: {setting}")

    net = Qwen3VLForConditionalGeneration.from_pretrained(
        recipe["model_id"],
        revision=recipe["revision"],
        dtype=DTYPE,
        device_map={"": 0},
        attn_implementation="sdpa"
    )
    if setting == "llm_lora":
        net = PeftModel.from_pretrained(net, ADAPTER, is_trainable=False)

    return net.eval()


@torch.no_grad()
def predict_sample(net, sample):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": sample["question"]}
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    with Image.open(sample["image_path"]) as source:
        image = source.convert("RGB")

    batch = processor(
        text=[text], images=[image], images_kwargs=EVAL_BUDGET,
        add_special_tokens=False, return_tensors="pt"
    ).to(DEVICE)

    config = GenerationConfig(
        do_sample=False,
        use_cache=True,
        max_new_tokens=CAPS[sample["task"]],
        eos_token_id=processor.tokenizer.eos_token_id,
        pad_token_id=processor.tokenizer.pad_token_id
    )

    with torch.autocast("cuda", dtype=DTYPE):
        output = net.generate(**batch, generation_config=config, use_model_defaults=False)

    new_ids = output[0, batch["input_ids"].shape[1]:]
    return processor.tokenizer.decode(new_ids, skip_special_tokens=True)

**OCR, Grounding and Receipt Extraction**